In [1]:
import pandas as pd
from pathlib import Path
import csv

base_dir = Path.cwd().parent
raw_dir = base_dir / "data/raw"
processed_dir = base_dir / "data/processed"

dfs = {}

dtype_config = {
    "customers": {
        "customer_zip_code_prefix": "str"
    },
    "geolocation": {
        "geolocation_zip_code_prefix": "str"
    },
    "sellers": {
        "seller_zip_code_prefix": "str"
    },
    "products": {
        "product_name_lenght": "Int64",
        "product_description_lenght": "Int64",
        "product_photos_qty": "Int64"
    }
}

datetime_config = {
    "orders": [
        "order_purchase_timestamp",
        "order_approved_at",
        "order_delivered_carrier_date",
        "order_delivered_customer_date",
        "order_estimated_delivery_date"
    ],
    "order_items": [
        "shipping_limit_date"
    ],
    "order_reviews": [
        "review_creation_date",
        "review_answer_timestamp"
    ]
}

for file in raw_dir.glob("*.csv"):
    name = file.stem.replace("olist_", "").replace("_dataset", "")
    dfs[name] = pd.read_csv(
        file,
        dtype=dtype_config.get(name, {})
    )
    if name in datetime_config:
        for col in datetime_config[name]:
            dfs[name][col] = pd.to_datetime(dfs[name][col])

orders = dfs["orders"]
customers = dfs["customers"]
geolocation = dfs["geolocation"]
sellers = dfs["sellers"]
products = dfs["products"]
category_name_translation = dfs["product_category_name_translation"]
order_items = dfs["order_items"]
order_payments = dfs["order_payments"]
order_reviews = dfs["order_reviews"]

# Orders

I valori mancanti in *order_approved_at* sono corretti, in quanto corrispondenti ad ordini:

* *canceled*, quindi con *order_delivered_carrier_date* e *order_delivered_customer_date* mancanti
* *created*, quindi con *order_delivered_carrier_date* e *order_delivered_customer_date* mancanti
* *delivered*, quindi con *order_delivered_carrier_date* e *order_delivered_customer_date* valorizzati

Anche i valori mancanti in *order_delivered_carrier_date* e *order_delivered_customer_date* risultano coerenti.

Le seguenti incongruenze temporali tra colonne sono mantenute, in quanto non si riesce a stabilire quali dati siano corretti:

* 166 ordini con *order_purchase_timestamp* > *order_delivered_carrier_date*
* 1359 ordini con *order_approved_at* > *order_delivered_carrier_date*
* 23 ordini con *order_delivered_carrier_date* > *order_delivered_customer_date*

In [2]:
orders.query(
    "order_approved_at.isnull()"
)

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
1130,00b1cb0320190ca0daa2c88b35206009,3532ba38a3fd242259a514ac2b6ae6b6,canceled,2018-08-28 15:26:39,NaT,NaT,NaT,2018-09-12
1801,ed3efbd3a87bea76c2812c66a0b32219,191984a8ba4cbb2145acb4fe35b69664,canceled,2018-09-20 13:54:16,NaT,NaT,NaT,2018-10-17
1868,df8282afe61008dc26c6c31011474d02,aa797b187b5466bc6925aaaa4bb3bed1,canceled,2017-03-04 12:14:30,NaT,NaT,NaT,2017-04-10
2029,8d4c637f1accf7a88a4555f02741e606,b1dd715db389a2077f43174e7a675d07,canceled,2018-08-29 16:27:49,NaT,NaT,NaT,2018-09-13
2161,7a9d4c7f9b068337875b95465330f2fc,7f71ae48074c0cfec9195f88fcbfac55,canceled,2017-05-01 16:12:39,NaT,NaT,NaT,2017-05-30
...,...,...,...,...,...,...,...,...
97696,5a00b4d35edffc56b825c3646a99ba9d,6a3bdf004ca96338fb5fad1b8d93c2e6,canceled,2017-07-02 15:38:46,NaT,NaT,NaT,2017-07-25
98415,227c804e2a44760671a6a5697ea549e4,62e7477e75e542243ee62a0ba73f410f,canceled,2017-09-28 15:02:56,NaT,NaT,NaT,2017-10-16
98909,e49e7ce1471b4693482d40c2bd3ad196,e4e7ab3f449aeb401f0216f86c2104db,canceled,2018-08-07 11:16:28,NaT,NaT,NaT,2018-08-10
99283,3a3cddda5a7c27851bd96c3313412840,0b0d6095c5555fe083844281f6b093bb,canceled,2018-08-31 16:13:44,NaT,NaT,NaT,2018-10-01


In [3]:
orders.query(
    "order_approved_at.isnull() and order_status == 'canceled'"
).info()

<class 'pandas.DataFrame'>
Index: 141 entries, 1130 to 99347
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype         
---  ------                         --------------  -----         
 0   order_id                       141 non-null    str           
 1   customer_id                    141 non-null    str           
 2   order_status                   141 non-null    str           
 3   order_purchase_timestamp       141 non-null    datetime64[us]
 4   order_approved_at              0 non-null      datetime64[us]
 5   order_delivered_carrier_date   0 non-null      datetime64[us]
 6   order_delivered_customer_date  0 non-null      datetime64[us]
 7   order_estimated_delivery_date  141 non-null    datetime64[us]
dtypes: datetime64[us](5), str(3)
memory usage: 9.9 KB


In [4]:
orders.query(
    "order_approved_at.isnull() and order_status != 'canceled'"
).sort_values("order_status")

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
7434,b5359909123fa03c50bdb0cfed07f098,438449d4af8980d107bf04571413a8e7,created,2017-12-05 01:07:52,NaT,NaT,NaT,2018-01-11
9238,dba5062fbda3af4fb6c33b1e040ca38f,964a6df3d9bdf60fe3e7b8bb69ed893a,created,2018-02-09 17:21:04,NaT,NaT,NaT,2018-03-07
21441,7a4df5d8cff4090e541401a20a22bb80,725e9c75605414b21fd8c8d5a1c2f1d6,created,2017-11-25 11:10:33,NaT,NaT,NaT,2017-12-12
58958,90ab3e7d52544ec7bc3363c82689965f,7d61b9f4f216052ba664f22e9c504ef1,created,2017-11-06 13:12:34,NaT,NaT,NaT,2017-12-01
55086,35de4050331c6c644cddc86f4f2d0d64,4ee64f4bfc542546f422da0aeb462853,created,2017-12-05 01:07:58,NaT,NaT,NaT,2018-01-08
5323,e04abd8149ef81b95221e88f6ed9ab6a,2127dc6603ac33544953ef05ec155771,delivered,2017-02-18 14:40:00,NaT,2017-02-23 12:04:47,2017-03-01 13:25:33,2017-03-17
67697,88083e8f64d95b932164187484d90212,f67cd1a215aae2a1074638bbd35a223a,delivered,2017-02-18 22:49:19,NaT,2017-02-22 11:31:06,2017-03-02 12:06:06,2017-03-21
63052,51eb2eebd5d76a24625b31c33dd41449,07a2a7e0f63fd8cb757ed77d4245623c,delivered,2017-02-18 15:52:27,NaT,2017-02-23 03:09:14,2017-03-07 13:57:47,2017-03-29
61743,2eecb0d85f281280f79fa00f9cec1a95,a3d3c38e58b9d2dfb9207cab690b6310,delivered,2017-02-17 17:21:55,NaT,2017-02-22 11:42:51,2017-03-03 12:16:03,2017-03-20
48401,7002a78c79c519ac54022d4f8a65e6e8,d5de688c321096d15508faae67a27051,delivered,2017-01-19 22:26:59,NaT,2017-01-27 11:08:05,2017-02-06 14:22:19,2017-03-16


In [5]:
orders.query(
    "order_purchase_timestamp > order_delivered_carrier_date"
)["order_status"].value_counts()

order_status
delivered    165
shipped        1
Name: count, dtype: int64

In [6]:
orders.query(
    "order_approved_at > order_delivered_carrier_date"
)["order_status"].value_counts()

order_status
delivered    1350
shipped         9
Name: count, dtype: int64

In [7]:
orders.query(
    "order_delivered_carrier_date > order_delivered_customer_date"
)["order_status"].value_counts()

order_status
delivered    23
Name: count, dtype: int64

In [8]:
orders.to_csv(
    processed_dir / "clean_orders.csv",
    sep="\t",
    float_format="%.6f",
    index=False,
    encoding="utf-8-sig",
    quoting=csv.QUOTE_ALL
)

# Customers

Dal profiling non sono emerse anomalie o trasformazioni da applicare al dataset.

In [9]:
customers.to_csv(
    processed_dir / "clean_customers.csv",
    sep="\t",
    float_format="%.6f",
    index=False,
    encoding="utf-8-sig",
    quoting=csv.QUOTE_ALL
)

# Geolocation

Le righe duplicate sono state eliminate.

I valori negativi delle colonne *geolocation_lat* e *geolocation_lng* sono coerenti con il dataset.

In [26]:
geolocation.drop_duplicates(inplace=True)

geolocation.duplicated().sum()

np.int64(0)

In [11]:
geolocation.to_csv(
    processed_dir / "clean_geolocation.csv",
    sep="\t",
    float_format="%.6f",
    index=False,
    encoding="utf-8-sig",
    quoting=csv.QUOTE_ALL
)

# Sellers

Dal profiling non sono emerse anomalie o trasformazioni da applicare al dataset.

In [12]:
sellers.to_csv(
    processed_dir / "clean_sellers.csv",
    sep="\t",
    float_format="%.6f",
    index=False,
    encoding="utf-8-sig",
    quoting=csv.QUOTE_ALL
)

# Products

I valori mancanti nelle colonne non sono stati sostituiti in quanto rappresentano incompletezza nel catalogo prodotti e non errori del dataset.

L'errore ortografico nelle colonne *product_name_lenght* e *product_description_lenght* è stato corretto.

In [13]:
products.rename(
    columns={"product_name_lenght": "product_name_length",
            "product_description_lenght": "product_description_length"},
    inplace=True    
)

products.info()

<class 'pandas.DataFrame'>
RangeIndex: 32951 entries, 0 to 32950
Data columns (total 9 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   product_id                  32951 non-null  str    
 1   product_category_name       32341 non-null  str    
 2   product_name_length         32341 non-null  Int64  
 3   product_description_length  32341 non-null  Int64  
 4   product_photos_qty          32341 non-null  Int64  
 5   product_weight_g            32949 non-null  float64
 6   product_length_cm           32949 non-null  float64
 7   product_height_cm           32949 non-null  float64
 8   product_width_cm            32949 non-null  float64
dtypes: Int64(3), float64(4), str(2)
memory usage: 2.4 MB


In [14]:
products.to_csv(
    processed_dir / "clean_products.csv",
    sep="\t",
    float_format="%.6f",
    index=False,
    encoding="utf-8-sig",
    quoting=csv.QUOTE_ALL
)

# Category Name Translation

Non sono emerse anomalie o trasformazioni da applicare al dataset. Le 2 *category_name_translation* mancanti verranno gestite successivamente.

In [15]:
category_name_translation.to_csv(
    processed_dir / "clean_category_name_translation.csv",
    sep="\t",
    float_format="%.6f",
    index=False,
    encoding="utf-8-sig",
    quoting=csv.QUOTE_ALL
)

# Order Items

Dal profiling non sono emerse anomalie o trasformazioni da applicare al dataset.

In [16]:
order_items.to_csv(
    processed_dir / "clean_order_items.csv",
    sep="\t",
    float_format="%.6f",
    index=False,
    encoding="utf-8-sig",
    quoting=csv.QUOTE_ALL
)

# Order Payments

Le anomalie rilevate nelle seguenti colonne non sono state modificate, in quanto non è possibile determinare con certezza i valori corretti:

* sequenze incoerenti nella colonna *payment_sequential*
* valori *not_defined* nella colonna *payment_type*
* valori incoerenti nella colonna *payment_installments*

In [17]:
(
    order_payments
        .groupby("order_id")["payment_sequential"]
        .apply(
            lambda x: sorted(x.tolist()) != list(range(1, x.max() + 1))
        )
).sum()

np.int64(80)

In [18]:
pd.merge(
    left=orders, right=order_payments, how="inner"
).query(
    "payment_type == 'not_defined'"    
)[["order_id", "order_status", "payment_type"]]

,order_id,order_status,payment_type
1175,00b1cb0320190ca0daa2c88b35206009,canceled,not_defined
41784,4637ca194b6387e2d538dc89b124b0ee,canceled,not_defined
42112,c8c528189310eaa44a745b8d9d26908b,canceled,not_defined


In [19]:
pd.merge(
    left=orders, right=order_payments, how="inner"
).query(
    "payment_installments < 1"
)

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,payment_sequential,payment_type,payment_installments,payment_value
66606,744bade1fcf9ff3f31d860ace076d422,5e5794daaa13f73e2f1cdb4114529843,delivered,2018-04-22 11:34:42,2018-04-24 19:04:46,2018-04-24 03:14:34,2018-04-27 20:55:28,2018-05-16,2,credit_card,0,58.69
69302,1a57108394169c0b47d8f876acc9ba2d,48ebb06cf56dba9d009230cc751bb195,delivered,2018-05-15 16:25:14,2018-05-15 16:36:52,2018-05-17 12:37:00,2018-05-24 15:45:41,2018-06-06,2,credit_card,0,129.94


In [20]:
order_payments.to_csv(
    processed_dir / "clean_order_payments.csv",
    sep="\t",
    float_format="%.6f",
    index=False,
    encoding="utf-8-sig",
    quoting=csv.QUOTE_ALL
)

# Order Reviews

I valori mancanti nelle colonne *review_comment_title* e *review_comment_message* sono stati mantenuti, in quanto l'assenza di un titolo o di un messaggio non compromette la validità della recensione rilasciata.

In [21]:
order_reviews.query(
    "review_comment_title.isna() | review_comment_message.isna()"
)

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
0,7bc2406110b926393aa56f80a40eba40,73fc7af87114b39712e6da79b0a377eb,4,NaN,NaN,2018-01-18,2018-01-18 21:46:59
1,80e641a11e56f04c1ad469d5645fdfde,a548910a1c6147796b98fdf73dbeba33,5,NaN,NaN,2018-03-10,2018-03-11 03:05:13
2,228ce5500dc1d8e020d8d1322874b6f0,f9e4b658b201a9f2ecdecbb34bed034b,5,NaN,NaN,2018-02-17,2018-02-18 14:36:24
3,e64fb393e7b32834bb789ff8bb30750e,658677c97b385a9be170737859d3511b,5,NaN,Recebi bem antes do prazo estipulado.,2017-04-21,2017-04-21 22:02:06
4,f7c4243c7fe1938f181bec41a392bdeb,8e6bfb81e283fa7e4f11123a3fb894f1,5,NaN,Parabéns lojas lannister adorei comprar pela I...,2018-03-01,2018-03-02 10:26:53
...,...,...,...,...,...,...,...
99219,574ed12dd733e5fa530cfd4bbf39d7c9,2a8c23fee101d4d5662fa670396eb8da,5,NaN,NaN,2018-07-07,2018-07-14 17:18:30
99220,f3897127253a9592a73be9bdfdf4ed7a,22ec9f0669f784db00fa86d035cf8602,5,NaN,NaN,2017-12-09,2017-12-11 20:06:42
99221,b3de70c89b1510c4cd3d0649fd302472,55d4004744368f5571d1f590031933e4,5,NaN,"Excelente mochila, entrega super rápida. Super...",2018-03-22,2018-03-23 09:10:43
99222,1adeb9d84d72fe4e337617733eb85149,7725825d039fc1f0ceb7635e3f7d9206,4,NaN,NaN,2018-07-01,2018-07-02 12:59:13


In [22]:
order_reviews.to_csv(
    processed_dir / "clean_order_reviews.csv",
    sep="\t",
    float_format="%.6f",
    index=False,
    encoding="utf-8-sig",
    quoting=csv.QUOTE_ALL
)